# La part du dollar dans la hausse · *The dollar's share of the rise*

Notebook compagnon de l'enquête **L'or monte-t-il parce que les monnaies s'effondrent ?** — [lire l'article](https://nmlab.io/ressources/prix-de-l-or-et-effondrement-des-monnaies).
Companion notebook to the study **Is gold rising because currencies are collapsing?**.

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données publiques du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's public data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


import io
import re
import urllib.request
from functools import lru_cache

import numpy as np
import pandas as pd
from pandas import DataFrame, Series

CMO_PAGE = "https://www.worldbank.org/en/research/commodity-markets"
CMO_FILE = ("https://thedocs.worldbank.org/en/doc/74e8be41ceb20fa0da750cda2f6b9e4e-0050012026"
            "/related/CMO-Historical-Data-Monthly.xlsx")
FRED_CSV = "https://fred.stlouisfed.org/graph/fredgraph.csv?id={}"

# H.10 : EUR, GBP et AUD sont cotés en dollars par unité étrangère — on les inverse
# pour obtenir partout des unités locales par dollar, comme dans l'article.
# H.10 quotes EUR, GBP and AUD as dollars per foreign unit: invert them so every
# series is local units per dollar, as in the article.
FX = {"EUR": ("DEXUSEU", True), "JPY": ("DEXJPUS", False), "GBP": ("DEXUSUK", True),
      "CHF": ("DEXSZUS", False), "CAD": ("DEXCAUS", False), "AUD": ("DEXUSAL", True),
      "CNY": ("DEXCHUS", False)}


def _fetch(url: str, tries: int = 5, headers: dict | None = None,
           data: str | None = None) -> bytes:
    """Télécharge une URL, avec reprises. Trois pièges de ces diffuseurs publics :
    la connexion peut être coupée sans raison, un quota (HTTP 429) peut s'appliquer quand
    plusieurs personnes partagent la même adresse — d'où l'attente plus longue — et une
    panne passagère (HTTP 5xx) se résout d'elle-même en quelques secondes.
    Download with retries: dropped connections, HTTP 429 quotas on shared addresses, and
    transient HTTP 5xx outages that clear on their own.
    """
    import time
    for attempt in range(tries):
        try:
            request = urllib.request.Request(url, headers=headers or {},
                                             data=data.encode() if data else None)
            return urllib.request.urlopen(request, timeout=120).read()
        except urllib.error.HTTPError as error:
            if (error.code != 429 and error.code < 500) or attempt == tries - 1:
                raise
            wait = (20 if error.code == 429 else 5) * (attempt + 1)
            reason = "quota atteint" if error.code == 429 else f"panne passagère ({error.code})"
            print(f"[or] {reason} chez le diffuseur, nouvelle tentative dans {wait} s…")
            time.sleep(wait)
        except Exception:
            if attempt == tries - 1:
                raise
            time.sleep(3 * (attempt + 1))
    raise RuntimeError("unreachable")


@lru_cache(maxsize=None)
def load_gold_usd() -> Series:
    """Or en dollars par once, moyennes mensuelles depuis 1960.

    Source : « Commodity Price Data » (Pink Sheet) de la Banque mondiale, feuille
    « Monthly Prices », colonne Gold — le fixing de Londres, en accès libre.
    World Bank Pink Sheet, monthly London gold price in US dollars per troy ounce.
    """
    try:
        raw = _fetch(CMO_FILE)
    except Exception:                                  # millésime renouvelé : on relit le lien
        page = _fetch(CMO_PAGE).decode("utf-8", "ignore")
        link = re.search(r"https://[^\"']*CMO-Historical-Data-Monthly\.xlsx", page)
        raw = _fetch(link.group(0))
    table = pd.read_excel(io.BytesIO(raw), sheet_name="Monthly Prices", skiprows=4)
    table = table.rename(columns={table.columns[0]: "date"})[["date", "Gold"]].dropna()
    dates = pd.to_datetime(table["date"].str.replace("M", "-"), format="%Y-%m")
    return Series(table["Gold"].values, index=dates).astype(float)


@lru_cache(maxsize=None)
def load_fred(series_id: str) -> Series:
    """Série FRED (CSV public, sans clé) ramenée à des moyennes mensuelles.
    A FRED series (public CSV, no key) averaged to monthly frequency."""
    table = pd.read_csv(io.StringIO(_fetch(FRED_CSV.format(series_id)).decode()))
    values = pd.to_numeric(table[table.columns[1]], errors="coerce")
    series = Series(values.values, index=pd.to_datetime(table[table.columns[0]])).dropna()
    return series.resample("MS").mean()


def load_gold_in_currencies(start: str, end: str) -> DataFrame:
    """Prix de l'or dans les huit devises du panier, mois par mois.

    Chaque prix local est le produit de la moyenne mensuelle de l'or en dollars
    et de la moyenne mensuelle du taux de change — l'ordre des opérations retenu
    par l'article. Le dollar vaut 1 par construction.
    Gold priced in the eight basket currencies, month by month.
    """
    gold = load_gold_usd()
    prices = {"USD": gold}
    for code, (series_id, invert) in FX.items():
        rate = load_fred(series_id)
        prices[code] = gold * (1 / rate if invert else rate)
    return DataFrame(prices).loc[start:end].dropna()


def effective_index(prices: DataFrame) -> Series:
    """Indice or effectif : moyenne géométrique équipondérée des huit prix locaux,
    base 100 au premier mois. Seule la moyenne géométrique garantit que l'indice
    des devises mesurées contre l'or est exactement l'inverse de celui-ci.
    Equal-weighted geometric mean of the eight local prices, first month = 100.
    """
    return 100 * np.exp(np.log(prices / prices.iloc[0]).mean(axis=1))


from matplotlib.figure import Figure
from matplotlib.ticker import FuncFormatter

START, END = "1999-01-01", "2026-05-01"

LABELS = {
    "fr": dict(
        title="Presque rien de la hausse ne vient du dollar",
        sub="Croissance cumulée en logarithmes, exprimée en pourcentage — décomposition exacte, pas une estimation.",
        usd="Prix de l'or en dollars", eff="Indice or effectif (les huit devises)",
        fx="Composante dollar contre le panier", share="{:.0f} % de la hausse",
        note="Identité comptable : prix en dollars = indice or effectif + composante dollar. Aucune approximation,\n"
             "aucune cause démontrée. Sources : Banque mondiale (or) ; Réserve fédérale, H.10 (change)."),
    "en": dict(
        title="Almost none of the rise comes from the dollar",
        sub="Cumulative log growth, shown in percent — an exact decomposition, not an estimate.",
        usd="Gold price in dollars", eff="Effective gold index (the eight currencies)",
        fx="Dollar component against the basket", share="{:.0f}% of the rise",
        note="Accounting identity: dollar price = effective gold index + dollar component. No approximation, and no\n"
             "cause established. Sources: World Bank (gold); Federal Reserve, H.10 (exchange rates)."),
}


def build_figure(prices: DataFrame, lang: str) -> Figure:
    """Le prix en dollars et ses deux composantes comptables, en croissance logarithmique."""
    text = LABELS[lang]
    usd = 100 * np.log(prices["USD"] / prices["USD"].iloc[0])
    eff = 100 * np.log(effective_index(prices) / 100)
    fx = usd - eff                     # ce que le dollar apporte face au panier

    fig = nm.figure(height_px=1120)
    ax = nm.axes(fig, left=0.075, right=0.982)
    ax.axhline(0, color=nm.COLORS["edge"], lw=2, zorder=1)
    ax.plot(usd.index, usd, color=nm.COLORS["text"], lw=3.4, zorder=5, label=text["usd"])
    ax.plot(eff.index, eff, color=nm.COLORS["amber"], lw=3.0, zorder=4, label=text["eff"])
    ax.plot(fx.index, fx, color=nm.COLORS["blue"], lw=3.0, zorder=4, label=text["fx"])
    ax.fill_between(fx.index, 0, fx, color=nm.COLORS["blue"], alpha=0.16, zorder=2)

    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.0f} %".replace(",", " ")
                                               if lang == "fr" else f"{v:,.0f}%"))
    ax.set_ylim(-40, 330)
    legend = ax.legend(loc="upper left", frameon=False, fontsize=20.5, labelcolor="linecolor",
                       handlelength=1.6, borderaxespad=1.2)
    for handle in legend.get_lines():
        handle.set_linewidth(3.4)

    share = 100 * float(fx.iloc[-1] / usd.iloc[-1])
    ax.annotate(text["share"].format(share),
                xy=(fx.index[-1], float(fx.iloc[-1])),
                xytext=(fx.index[int(len(fx) * 0.72)], 78),
                color=nm.COLORS["blue"], fontsize=21, fontweight="bold", ha="center",
                arrowprops=dict(arrowstyle="-", color=nm.COLORS["blue"], lw=2, alpha=0.7))

    nm.header(fig, text["title"], text["sub"])
    nm.footer(fig, text["note"])
    return fig


build_figure(load_gold_in_currencies(START, END), LANG)